# **Notebook for EDA Work done for Project**
by Jack Phelan

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Data Quality Assessment](#data-quality-assessment)
3. [Statistical Summary of Dataset](#statistical-summary-of-dataset)
4. [Univariate Analysis](#univariate-analysis)
5. [Bivariate Analysis](#bivariate-analysis)

In [ ]:
# imports
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)

In [ ]:
raw_df = pd.read_csv(
    "../data/raw/healthcare_readmissions_dataset_train.csv",
    keep_default_na=False,
    na_values=[""],
)

In [ ]:
raw_df.head()

## **1. Dataset Overview**
- Dataset source, number of records (rows) and variables (columns), file format, any metadata or documentation available


In [ ]:
raw_df_rows = raw_df.shape[0]
raw_df_cols = raw_df.shape[1]
print(f"Dataset contains {raw_df_rows} records and {raw_df_cols} variables.")

## **2. Data Quality Assessment**
- Summarize the quality and completeness of the raw data
- Identify unusual data points or distributions (outliers)


In [ ]:
display(raw_df.info())

In [ ]:
# rename columns for consistency


def format_columns(df: pd.DataFrame) -> pd.DataFrame:
    df_transformed = df.copy()

    if "PatientID" in df_transformed.columns:
        df_transformed = df_transformed.rename(
            columns={"PatientID": "patient_id"}
        )  # manual change to align

    df_transformed.columns = (
        df_transformed.columns.str.strip().str.lower().str.replace(" ", "_")
    )
    df_transformed.columns = df_transformed.columns.str.replace("(", "").str.replace(
        ")", ""
    )

    return df_transformed

In [ ]:
df_transformed = format_columns(raw_df)

display(df_transformed.head())

In [ ]:
# looking at distribution
set_theme()

categories = [
    "hospital_id",
    "gender",
    "smoker",
    "has_diabetes",
    "has_hypertension",
    "exercise_frequency",
    "diet_type",
    "number_of_prior_visits",
    "medications_prescribed",
    "length_of_stay",
    "type_of_treatment",
    "ethnicity",
    "height_m",
    "readmission_within_30_days",
]

measures = ["bmi", "weight_kg", "adjusted_weight_kg", "age"]

In [ ]:
plot_grid(
    df=df_transformed,
    column_names=categories,
    plot_func=plot_barplot,
    palette="flare",
    n_plot_cols=3,
    sharey=True,
)

Categorical Distribution Notes:
- target class imbalanced
- medications prescribed and number of prior visits have a strong amount of missings (as well as 0 values)
- gender, hospital-id, both quite even distributions

In [ ]:
plot_grid(
    df=df_transformed,
    column_names=measures,
    plot_func=plot_histogram,
    n_plot_cols=2,
    sharey=True, 
    bins="auto",
    stat="count",
    kde=False,
    show_bin_counts=False,
)

In [ ]:
plot_histogram(
    df_transformed, column_name="age", bins="auto", kde=False, show_bin_counts=False
)

Measure Distribution Notes:
- lots of outliers in age - ages above 100 should not be possible
- potentially similar with weight; will relook once outliers of age are checked

In [ ]:
# age outlier checking
threshold = 3
age_z_scores = np.abs(stats.zscore(df_transformed["age"]))
age_outliers = np.where(age_z_scores > threshold)[0]
print(f"Identified {len(age_outliers)} age outliers at threshold {threshold}:")

display(df_transformed.loc[age_outliers, "age"])


In [ ]:
df_transformed_2 = df_transformed.drop(index=age_outliers).reset_index(drop=True)

In [ ]:
plot_grid(
    df=df_transformed_2,
    column_names=measures,
    plot_func=plot_histogram,
    n_plot_cols=2,
    sharey=True, 
    bins="auto",
    stat="count",
    kde=False,
    show_bin_counts=False,
)

age dsitribution fixed. some high weight values but those could be plausible, also adjusted weight looks normal as well as bmi distribution, so will leave weight as is for now.

In [ ]:
df_final = df_transformed_2.copy()

In [ ]:
display(df_final.info())

In [ ]:
df_final.describe()

In [ ]:
# latex cell
df_latex = df_final.copy().drop(columns=["patient_id"]).describe().transpose().round(2)

styled_latex = df_latex.style.to_latex(caption="Measure Summary Statistics",
    label="tab:metrics",
    hrules=True  # Adds \toprule, \
)
print(styled_latex)

## **4. Bivariate Analysis**
- Bivariate visualizations of each variable vs the response variable and any other “interesting” pairs of variables. Can be done with Tableau or Python
- Annotate any significant observations


In [ ]:
plot_categorical_x_categorical_grid(
    df=df_final,
    base_cat_col="readmission_within_30_days",  
    category_cols=["smoker", "has_diabetes", "exercise_frequency", "diet_type", "gender", "has_hypertension"],
    plot_func=sns.countplot,
    n_plot_cols=2,
    figsize_scale=(8, 5),
)

In [ ]:
plot_numeric_x_numeric_grid(
    df=df_final,
    base_numeric_col="bmi",
    numeric_cols=["age", "weight_kg", "adjusted_weight_kg", "length_of_stay"],
    plot_func='hexbin',
    n_plot_cols=2,
    gridsize=25,
    cmap="magma",
)

In [ ]:
# correlation matrix

numeric_df = df_final[measures]

corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# BMI vs categorical features
plot_numeric_x_across_categories_grid(
    df=df_final,
    numeric_col="bmi",
    category_cols=["gender", "smoker", "has_diabetes", "exercise_frequency", "diet_type", "hospital_id", "type_of_treatment"],
    plot_func=sns.boxplot,
    n_plot_cols=2,
    figsize_scale=(8, 5),
)


In [ ]:
# Plot all numeric columns as boxplots split by target
plot_all_numeric_by_base_category_grid(
    df=df_final,
    base_cat_col="readmission_within_30_days",
    plot_func=sns.boxplot,
    n_plot_cols=3,
    figsize_scale=(6, 4),
)

In [ ]:
df_final.duplicated().sum()

In [ ]:
df_final.to_csv("../data/processed/healthcare_readmissions_dataset_train_processed.csv", index=False)

In [ ]:
"""plot_grid(
    df=df_final,
    column_names=categories,
    plot_func=plot_barplot,
    palette="flare",
    n_plot_cols=3,
    sharey=True,
)"""

In [ ]:
"""# individual histograms
## numerics

for measure in measures:
    plot_histogram(
        df_final, column_name=measure, bins="auto", kde=False, show_bin_counts=False
    )"""

In [ ]:
"""# individual category distributions

for category in categories:
    plot_barplot(
        df_final, column_name=category, palette="flare"
    )
"""

In [ ]:
plot_categorical_x_categorical_grid(
    df=df_final,
    base_cat_col="readmission_within_30_days",  
    category_cols=[cat for cat in categories if cat != "readmission_within_30_days"],
    plot_func=sns.countplot,
    n_plot_cols=3,
    figsize_scale=(8, 5),
    
)

In [ ]:
# Plot all numeric columns as boxplots split by target
plot_all_numeric_by_base_category_grid(
    df=df_final,
    base_cat_col="readmission_within_30_days",
    numeric_cols=[num for num in measures],
    plot_func=sns.boxplot,
    n_plot_cols=2,
    figsize_scale=(6, 4),
)

# correlation matirx

In [ ]:
df_final.head()

In [ ]:
columns_to_encode = [cat for cat in categories if cat != "readmission_within_30_days"]
columns_to_encode.remove("medications_prescribed")
columns_to_encode.remove("number_of_prior_visits")
columns_to_encode.remove("length_of_stay")
columns_to_encode.remove("height_m")

print(columns_to_encode)




In [ ]:
df_final_ohe = pd.get_dummies(df_final, columns=columns_to_encode, drop_first=True)
df_final_ohe.drop(columns=["patient_id", "readmission_within_30_days"], inplace=True)

In [ ]:
corr = df_final_ohe.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of All Features (One-Hot Encoded)")
plt.tight_layout()
plt.show()

In [ ]:
# individual category vs output graphs

for category in categories:
    if category != "readmission_within_30_days":
        plot_barplot(
            df_final,
            column_name=category,
            palette="flare",
            hue="readmission_within_30_days",
        )


In [ ]:
# output dataset

df_final.to_csv("../data/interim/healthcare_readmissions_dataset_train_post_eda.csv", index=False)
